In [ ]:
import random
import networkx as nx
import pandas as pd

def simulate_probabilistic_propagation(
    G,
    risk_df,
    start_node=None,
    steps=10,
    base_infection_prob=0.3   # base probability — scaled by transfer volume
):
    """
    Probabilistic propagation model where:
    - Initial infection targets highest in-degree hospital
    - Spread probability weighted by transfer volume (information sharing intensity)
    - Mirrors malware propagation via data sharing connections
    """

    # ── Stage 1: Select initial infection target ──────────────
    if start_node is None:
        # Highest in-degree = largest information receiving surface
        start_node = risk_df.sort_values("in_degree", ascending=False).index[0]

    print(f"Initial infection: {start_node}")
    print(f"In-degree centrality: {risk_df.loc[start_node, 'in_degree']:.4f}")

    infected   = {start_node}
    timeline   = [{"step": 0, "newly_infected": {start_node}, "total_infected": 1}]

    # Max transfer weight for normalisation
    max_weight = max(
        G[u][v].get("weight", 1)
        for u, v in G.edges()
    )

    

# ── Stage 2: Probabilistic spread ─────────────────────────


In [ ]:
for step in range(1, steps + 1):
        newly_infected = set()

        for node in list(infected):
            # Get all neighbours this node can spread TO
            for neighbor in G.successors(node):
                if neighbor in infected:
                    continue

                # Infection probability weighted by transfer volume
                # More transfers = higher probability of malware spread
                transfer_weight = G[node][neighbor].get("weight", 1)
                infection_prob  = base_infection_prob * (transfer_weight / max_weight)

                # Cap at 0.95 to avoid certainty
                infection_prob  = min(infection_prob, 0.95)

                if random.random() < infection_prob:
                    newly_infected.add(neighbor)

        infected |= newly_infected
        timeline.append({
            "step":            step,
            "newly_infected":  newly_infected,
            "total_infected":  len(infected),
            "pct_infected":    len(infected) / G.number_of_nodes() * 100
        })

        print(f"  Step {step}: +{len(newly_infected)} hospitals "
              f"| Total: {len(infected)} ({len(infected)/G.number_of_nodes()*100:.1f}%)")

        # Stop if no new infections
        if not newly_infected:
            print(f"  Propagation stopped at step {step}")
            break

    return infected, pd.DataFrame(timeline)

# ── Run simulation ────────────────────────────────────────────
random.seed(42)  # reproducibility
infected_nodes, timeline_df = simulate_probabilistic_propagation(
    G,
    risk_df,
    steps=10,
    base_infection_prob=0.3
)

print(f"\nFinal: {len(infected_nodes)} / {G.number_of_nodes()} hospitals infected "
      f"({len(infected_nodes)/G.number_of_nodes()*100:.1f}%)")

In [ ]:
# For each hospital, simulate what happens if it is protected (removed from spread)
# Hospital whose protection most reduces final infection count = most critical

print("\nIdentifying critical protection targets...")
critical_results = []

baseline_infected, _ = simulate_probabilistic_propagation(G, risk_df, steps=10)
baseline_count = len(baseline_infected)

# Test protecting each high-risk hospital
candidates = risk_df.sort_values("propagation_risk_score", ascending=False).head(20).index

for hospital in candidates:
    # Remove hospital from graph (simulating protection/isolation)
    G_protected = G.copy()
    G_protected.remove_node(hospital)

    # Re-run simulation without this hospital
    risk_df_temp = risk_df.drop(index=hospital, errors="ignore")

    try:
        infected_protected, _ = simulate_probabilistic_propagation(
            G_protected, risk_df_temp, steps=10
        )
        infections_prevented = baseline_count - len(infected_protected)
    except Exception:
        infections_prevented = 0

    critical_results.append({
        "hospital":             hospital,
        "infections_prevented": infections_prevented,
        "pct_reduction":        infections_prevented / baseline_count * 100,
        "propagation_risk":     risk_df.loc[hospital, "propagation_risk_score"],
        "betweenness":          risk_df.loc[hospital, "betweenness"],
        "community":            risk_df.loc[hospital, "community"]
    })

critical_df = pd.DataFrame(critical_results) \
              .sort_values("infections_prevented", ascending=False)

print("\n--- Critical Protection Targets ---")
print("Protecting these hospitals prevents the most spread:\n")
print(critical_df[[
    "hospital",
    "infections_prevented",
    "pct_reduction",
    "propagation_risk",
    "betweenness",
    "community"
]].round(4).to_string(index=False))

Following initial infection of the highest in-degree hospital — reflecting the greatest information-receiving surface area — malware propagation was modelled probabilistically across directed transfer edges. Infection probability at each step was weighted by inter-hospital transfer volume, operationalising information-sharing intensity as a proxy for data pathway exploitation. Monte Carlo simulation (n=1,000 iterations, seed=42) was used to account for stochastic variation in spread. Critical nodes were identified as hospitals whose targeted protection produced the greatest reduction in total network infection, combining betweenness centrality with simulated infection prevention capacity.

In [ ]:
from tqdm import tqdm

n_simulations = 1000
all_timelines = []

for i in tqdm(range(n_simulations), desc="Monte Carlo"):
    _, tl = simulate_probabilistic_propagation(
        G, risk_df, steps=10, base_infection_prob=0.3
    )
    all_timelines.append(tl.set_index("step")["pct_infected"])

# Average across all simulations
mc_df = pd.concat(all_timelines, axis=1)
mc_df["mean"]  = mc_df.mean(axis=1)
mc_df["lower"] = mc_df.quantile(0.05, axis=1)  # 5th percentile
mc_df["upper"] = mc_df.quantile(0.95, axis=1)  # 95th percentile

print("\nMonte Carlo Results (mean % infected per step):")
print(mc_df[["mean", "lower", "upper"]].round(2).to_string())